In [3]:

from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = """
The Atlas Knowledge Management Platform is an internal enterprise system designed to help employees search, retrieve, and understand information distributed across company policies, engineering documentation, product manuals, operational procedures, security guidelines, and internal knowledge articles. The platform was introduced after the company experienced difficulties maintaining consistent access to information across multiple departments. Previously, employees relied on a combination of shared network drives, email conversations, manually maintained wiki pages, and document management systems. As the number of documents increased, employees frequently encountered outdated information, duplicate documents, and uncertainty about which document represented the current company policy. Atlas was therefore designed around a centralized document ingestion and retrieval architecture that allows authorized administrators to upload organizational documents while employees interact with the system through a natural-language question-answering interface.

The platform consists of six major subsystems: document ingestion, document processing, embedding generation, vector storage, retrieval, and response generation. The document ingestion subsystem accepts PDF, DOCX, Markdown, and plain-text documents. Once a document has been uploaded, the ingestion service assigns it a unique document identifier and records metadata including the original filename, department, document owner, upload timestamp, document version, and access classification. The document is then passed to the processing subsystem, which extracts textual content, removes unnecessary formatting artifacts, identifies page boundaries, and preserves important structural information such as headings, paragraphs, tables, and lists. The resulting representation is divided into smaller sections before being converted into vector representations for semantic retrieval.

One of the most important architectural decisions in Atlas was to separate document processing from document retrieval. Document processing is performed asynchronously because large documents can require significant CPU and memory resources. When an administrator uploads a document, the API immediately stores the document metadata and creates an ingestion job rather than processing the entire document during the HTTP request. A background worker retrieves the job, downloads the original file from object storage, extracts its content, generates chunks, creates embeddings, and stores the resulting records in the vector database. This approach prevents long-running ingestion operations from blocking API workers and allows the organization to process multiple documents concurrently.

The platform uses PostgreSQL as its primary relational database and the pgvector extension for semantic vector search. PostgreSQL stores document metadata, user information, access-control information, ingestion status, and references to individual document chunks. Each chunk contains the textual content, document identifier, page number, section title, chunk position, document version, and embedding vector. The embedding vector is generated by an embedding model and represents the semantic meaning of the chunk in a high-dimensional numerical space. The dimension of the vector is determined by the embedding model rather than by the size of the document. Consequently, every vector stored in a particular vector column must have the same dimensionality as required by the selected embedding model.

Document Ingestion:

When an administrator uploads a document, Atlas first validates the file type and size. The system does not immediately assume that a file is safe merely because its extension is .pdf or .docx. File signatures are checked where possible, and uploaded files are stored outside the application server in object storage. Each uploaded document receives a unique identifier generated by the application. The identifier is used throughout the ingestion pipeline so that the original document, processing jobs, chunks, embeddings, and audit records can be associated with the same logical document.

After successful upload, the document enters a PENDING state. A background worker then changes the state to PROCESSING and begins extraction. If extraction succeeds, the document moves to CHUNKING, followed by EMBEDDING, and finally COMPLETED. If any stage fails, the document moves to FAILED, and the failure reason is recorded. Failed documents can be retried without requiring the administrator to upload the original file again. The retry mechanism is particularly important because external embedding providers can temporarily return rate-limit or availability errors.

The ingestion pipeline is designed to be idempotent. If the same ingestion job is accidentally delivered to a worker more than once, the system should not create duplicate chunks and duplicate vectors. Each chunk therefore has a deterministic identity derived from the document version and chunk position. Before inserting a chunk, the worker can verify whether that chunk already exists. This property is important because background job systems generally provide at-least-once delivery rather than guaranteeing exactly-once execution.

Document versions are treated as separate logical versions of the same document. Suppose the security team publishes version 4 of its password policy after version 3 has already been indexed. Atlas does not simply overwrite every record belonging to version 3. Instead, the new version receives a new document-version identifier. The old version can remain available for auditing while retrieval is configured to prefer the latest active version. This prevents employees from accidentally receiving outdated policy information while preserving historical records for compliance purposes.
"""
docs = splitter.create_documents([documents])
chunks = splitter.split_documents(docs)
print(chunks)

[Document(metadata={}, page_content='The Atlas Knowledge Management Platform is an internal enterprise system designed to help employees search, retrieve, and understand information distributed across company policies, engineering documentation, product manuals, operational procedures, security guidelines, and internal knowledge articles. The platform was introduced after the company experienced difficulties maintaining consistent access to information across multiple departments. Previously, employees relied on a combination of'), Document(metadata={}, page_content='Previously, employees relied on a combination of shared network drives, email conversations, manually maintained wiki pages, and document management systems. As the number of documents increased, employees frequently encountered outdated information, duplicate documents, and uncertainty about which document represented the current company policy. Atlas was therefore designed around a centralized document ingestion and retr

In [ ]:
%pip install -U langchain langchain-core langchain-google-genai langchain_text_splitters dotenv

In [4]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from google import genai
import os

load_dotenv(override=True)
apikey = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=apikey)
m = 'gemini-3.5-flash-lite'
gemini_model = ChatGoogleGenerativeAI(model=m, google_api_key=apikey, temperature=0.8)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=apikey)


In [ ]:
%pip install -U langchain-postgres psycopg2-binary


In [5]:
from langchain_postgres import PGVector

db_url = os.getenv("DB_URL")
vectorstore = PGVector.from_documents(
    documents= chunks,
    embedding=embeddings,
    connection=db_url,
    collection_name="atlas_docs"
)

Test similarity search

In [6]:
results = vectorstore.similarity_search("What is Atlas?", k=2)
print(results[0].page_content)

The Atlas Knowledge Management Platform is an internal enterprise system designed to help employees search, retrieve, and understand information distributed across company policies, engineering documentation, product manuals, operational procedures, security guidelines, and internal knowledge articles. The platform was introduced after the company experienced difficulties maintaining consistent access to information across multiple departments. Previously, employees relied on a combination of
